In [1]:
# 02_preprocessing.ipynb

# %% [markdown]
# # Cross-Domain AI Governance Gap Analysis
# ## 02. Preprocessing
#
# Prepare four datasets (credit + insurance) for a comparable cross-domain analysis:
# - a single shared preprocessing routine applied consistently to every dataset
# - numeric imputation (median) + scaling; categorical imputation ("missing") + one-hot
# - subsample very large datasets (e.g. Porto Seguro) to a common cap so that sample-size
#   differences do not confound the cross-domain governance-gap comparison
# - stratified train/test split; persist splits and metadata

# %%
# Imports and configuration
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")

# Global config
RANDOM_STATE = 42
TEST_SIZE = 0.25
# Cap on rows per dataset before splitting, to align sample sizes across domains.
# Set to None to disable. Chosen so the largest sets are comparable to the others
# while retaining enough positives for stable subgroup metrics.
MAX_ROWS = 40000

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

# %%
# Load frames produced by 01_data_loading
df_german = pd.read_parquet(os.path.join(PROCESSED_DIR, "german.parquet"))
df_home = pd.read_parquet(os.path.join(PROCESSED_DIR, "credit_card_approval.parquet"))
df_ins = pd.read_parquet(os.path.join(PROCESSED_DIR, "insurance.parquet"))
df_porto = pd.read_parquet(os.path.join(PROCESSED_DIR, "porto_seguro.parquet"))

print("german:", df_german.shape)
print("credit_card_approval:", df_home.shape)
print("insurance:", df_ins.shape)
print("porto_seguro:", df_porto.shape)

# %%
# Optional stratified subsampling to a common row cap (keeps class balance)
def subsample(df, target_col, max_rows=MAX_ROWS, random_state=RANDOM_STATE):
    if max_rows is None or len(df) <= max_rows:
        return df.copy()
    frac = max_rows / len(df)
    out = (
        df.groupby(target_col, group_keys=False)
          .apply(lambda g: g.sample(frac=frac, random_state=random_state))
    )
    return out.reset_index(drop=True)

df_porto_s = subsample(df_porto, target_col="target")
print("Porto Seguro after subsample:", df_porto_s.shape)
print("target dist:", df_porto_s["target"].value_counts(normalize=True).round(4).to_dict())

# %%
# Shared preprocessing helper (identical routine across all datasets for comparability).
# Splits first, then fits imputation/scaling/encoding on train only to avoid leakage.
def preprocess_dataset(
    df, target_col, sensitive_cols, drop_cols=None,
    test_size=TEST_SIZE, random_state=RANDOM_STATE
):
    df = df.copy()
    drop_cols = drop_cols or []

    y = df[target_col].astype(int)
    sensitive = df[sensitive_cols].copy()

    feature_df = df.drop(columns=[target_col] + sensitive_cols + drop_cols, errors="ignore")

    num_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()

    idx_train, idx_test = train_test_split(
        feature_df.index, test_size=test_size,
        random_state=random_state, stratify=y
    )

    X_train_raw = feature_df.loc[idx_train].copy()
    X_test_raw = feature_df.loc[idx_test].copy()

    # Numeric: median imputation (fit on train) + standard scaling
    num_medians = X_train_raw[num_cols].median()
    X_train_raw[num_cols] = X_train_raw[num_cols].fillna(num_medians)
    X_test_raw[num_cols] = X_test_raw[num_cols].fillna(num_medians)

    scaler = StandardScaler()
    X_train_num = pd.DataFrame(
        scaler.fit_transform(X_train_raw[num_cols]),
        columns=num_cols, index=X_train_raw.index
    ) if num_cols else pd.DataFrame(index=X_train_raw.index)
    X_test_num = pd.DataFrame(
        scaler.transform(X_test_raw[num_cols]),
        columns=num_cols, index=X_test_raw.index
    ) if num_cols else pd.DataFrame(index=X_test_raw.index)

    # Categorical: fill missing with "missing", one-hot, align train/test
    if cat_cols:
        X_train_cat = X_train_raw[cat_cols].astype("object").fillna("missing")
        X_test_cat = X_test_raw[cat_cols].astype("object").fillna("missing")
        X_train_cat = pd.get_dummies(X_train_cat, drop_first=False)
        X_test_cat = pd.get_dummies(X_test_cat, drop_first=False)
        X_train_cat, X_test_cat = X_train_cat.align(
            X_test_cat, join="left", axis=1, fill_value=0
        )
        X_test_cat = X_test_cat[X_train_cat.columns]
    else:
        X_train_cat = pd.DataFrame(index=X_train_raw.index)
        X_test_cat = pd.DataFrame(index=X_test_raw.index)

    X_train = pd.concat([X_train_num, X_train_cat], axis=1)
    X_test = pd.concat([X_test_num, X_test_cat], axis=1)

    train = X_train.copy()
    train[target_col] = y.loc[idx_train].values
    for c in sensitive_cols:
        train[c] = sensitive.loc[idx_train, c].values

    test = X_test.copy()
    test[target_col] = y.loc[idx_test].values
    for c in sensitive_cols:
        test[c] = sensitive.loc[idx_test, c].values

    meta = {
        "target_col": target_col,
        "sensitive_cols": sensitive_cols,
        "n_features": X_train.shape[1],
        "n_train": len(train),
        "n_test": len(test),
        "num_cols": num_cols,
        "cat_cols": cat_cols,
    }
    return train, test, meta

# %%
# German Credit [credit]
german_train, german_test, german_meta = preprocess_dataset(
    df_german,
    target_col="target",
    sensitive_cols=["sex", "age_group"],
    drop_cols=["personal_status_sex", "domain"],
)
print("German:", german_train.shape, german_test.shape, "| features:", german_meta["n_features"])

# %%
# Credit Card Approval [credit]
# Carry the 60dpd label for optional threshold sensitivity; ID/gender code dropped.
home_alt_target = df_home[["target_60dpd"]].copy()
home_train, home_test, home_meta = preprocess_dataset(
    df_home,
    target_col="target",  # equals target_30dpd (primary)
    sensitive_cols=["sex"],
    drop_cols=["ID", "CODE_GENDER", "target_30dpd", "target_60dpd", "domain"],
)
home_train = home_train.join(home_alt_target, how="left")
home_test = home_test.join(home_alt_target, how="left")
print("Credit Card Approval:", home_train.shape, home_test.shape, "| features:", home_meta["n_features"])

# %%
# Medical Cost [insurance]
# Drop the raw 'smoker' string and 'charges' (charges leaks the target); keep smoker_flag.
ins_train, ins_test, ins_meta = preprocess_dataset(
    df_ins,
    target_col="target",
    sensitive_cols=["sex", "age_group", "smoker_flag"],
    drop_cols=["smoker", "charges", "domain"],
)
print("Medical Cost:", ins_train.shape, ins_test.shape, "| features:", ins_meta["n_features"])

# %%
# Porto Seguro [insurance]
# No explicit sensitive attribute; proxy variables (ps_ind_* cat/bin) are treated as sensitive.
proxy_cols = [c for c in df_porto_s.columns if c.startswith("ps_ind") and ("cat" in c or "bin" in c)]
porto_train, porto_test, porto_meta = preprocess_dataset(
    df_porto_s,
    target_col="target",
    sensitive_cols=proxy_cols,
    drop_cols=["id", "domain"],
)
print("Porto Seguro:", porto_train.shape, porto_test.shape, "| features:", porto_meta["n_features"])

# %%
# Save processed splits
german_train.to_parquet(os.path.join(PROCESSED_DIR, "german_train.parquet"))
german_test.to_parquet(os.path.join(PROCESSED_DIR, "german_test.parquet"))
home_train.to_parquet(os.path.join(PROCESSED_DIR, "credit_card_approval_train.parquet"))
home_test.to_parquet(os.path.join(PROCESSED_DIR, "credit_card_approval_test.parquet"))
ins_train.to_parquet(os.path.join(PROCESSED_DIR, "insurance_train.parquet"))
ins_test.to_parquet(os.path.join(PROCESSED_DIR, "insurance_test.parquet"))
porto_train.to_parquet(os.path.join(PROCESSED_DIR, "porto_seguro_train.parquet"))
porto_test.to_parquet(os.path.join(PROCESSED_DIR, "porto_seguro_test.parquet"))

# Save metadata
metadata = {
    "german": {**german_meta, "domain": "credit"},
    "credit_card_approval": {**home_meta, "domain": "credit"},
    "insurance": {**ins_meta, "domain": "insurance"},
    "porto_seguro": {**porto_meta, "domain": "insurance"},
    "config": {"random_state": RANDOM_STATE, "test_size": TEST_SIZE, "max_rows": MAX_ROWS},
}
with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# %%
# Preprocessing summary table
prep_summary = pd.DataFrame([
    {"domain": "credit", "dataset": "German Credit", "n_features": german_meta["n_features"],
     "n_train": german_meta["n_train"], "n_test": german_meta["n_test"],
     "sensitive": ", ".join(german_meta["sensitive_cols"])},
    {"domain": "credit", "dataset": "Credit Card Approval", "n_features": home_meta["n_features"],
     "n_train": home_meta["n_train"], "n_test": home_meta["n_test"],
     "sensitive": ", ".join(home_meta["sensitive_cols"])},
    {"domain": "insurance", "dataset": "Medical Cost", "n_features": ins_meta["n_features"],
     "n_train": ins_meta["n_train"], "n_test": ins_meta["n_test"],
     "sensitive": ", ".join(ins_meta["sensitive_cols"])},
    {"domain": "insurance", "dataset": "Porto Seguro", "n_features": porto_meta["n_features"],
     "n_train": porto_meta["n_train"], "n_test": porto_meta["n_test"],
     "sensitive": f"{len(porto_meta['sensitive_cols'])} proxy cols"},
])
print(prep_summary.to_string(index=False))
prep_summary.to_csv(os.path.join(TAB_DIR, "02_preprocess_summary.csv"), index=False)

german: (1000, 24)
credit_card_approval: (36457, 23)
insurance: (1338, 11)
porto_seguro: (595212, 60)


C:\Users\User\AppData\Local\Temp\ipykernel_18848\3421855929.py:62: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(frac=frac, random_state=random_state))


Porto Seguro after subsample: (40000, 60)
target dist: {0: 0.9636, 1: 0.0365}
German: (750, 60) (250, 60) | features: 57
Credit Card Approval: (27342, 56) (9115, 56) | features: 53
Medical Cost: (1003, 11) (335, 11) | features: 7
Porto Seguro: (30000, 58) (10000, 58) | features: 43
   domain              dataset  n_features  n_train  n_test                   sensitive
   credit        German Credit          57      750     250              sex, age_group
   credit Credit Card Approval          53    27342    9115                         sex
insurance         Medical Cost           7     1003     335 sex, age_group, smoker_flag
insurance         Porto Seguro          43    30000   10000               14 proxy cols
